# Experiment: DBCC baseline

Runs the distance-based collaborative classification algorithm of Ghanaiem et al.
(*Mathematics* 11(23):4784, 2023; `src/dbcc.py`) over the same prediction JSONs,
evaluation sizes, trials and seeds as `run_experiment.ipynb`, and writes one record
per (dataset, size, trial) to a JSONL file. Each record carries the `subsample_hash`
of its subsample, so it joins one-to-one with `experiment_records.jsonl`
(`analysis/dbcc_comparison.ipynb` does the join and produces the comparison table).

Only hard predictions are read; logits, if present, are ignored. The run is
resumable: re-running the loop skips (dataset, size, trial) triples already in the
output file.

In [ ]:
import os
# Anchor to repo root so relative data/ and src/ paths resolve no matter where this runs.
while not os.path.isdir("data/results") and os.getcwd() != "/":
    os.chdir("..")
print("Working directory:", os.getcwd())


In [ ]:
# If running from the repo root, make src/ importable.
import sys, os
sys.path.append(os.path.join(os.getcwd(), "src"))

import json, glob, time
import numpy as np

from io_utils import load_dataset_predictions, make_test_sizes
from pipeline import build_subsample_indices, hash_indices
from metrics import summarize_method_performance
from dbcc import dbcc_predict

In [ ]:
# --- paths (edit these) ---
PREDICTIONS_DIR = "data/predictions"
OUTPUT_JSONL    = "data/results/dbcc_records.jsonl"

# --- protocol constants (match run_experiment.ipynb) ---
NUM_SIZES          = 20
TRIALS_PER_SIZE    = 3
ENFORCE_NUM_MODELS = 15

# --- DBCC settings (see src/dbcc.py) ---
MAX_EM_ITER        = 20      # cap on the correction loop; the paper gives none
RECORD_STAGE1_TOO  = True    # also record the stage-1-only variant (no correction loop)

os.makedirs(os.path.dirname(OUTPUT_JSONL), exist_ok=True)

In [ ]:
json_paths = sorted(glob.glob(os.path.join(PREDICTIONS_DIR, "*_test_predictions_logits.json")))
print(f"Found {len(json_paths)} prediction file(s) in {PREDICTIONS_DIR}")

done = set()
if os.path.exists(OUTPUT_JSONL):
    with open(OUTPUT_JSONL) as f:
        for line in f:
            r = json.loads(line); done.add((r["dataset"], r["size"], r["trial"]))
print(f"{len(done)} run(s) already recorded; they will be skipped")

with open(OUTPUT_JSONL, "a") as out:
    for path in json_paths:
        loaded = load_dataset_predictions(path, enforce_num_models=ENFORCE_NUM_MODELS, sanity_check=False)
        name, C = loaded["dataset"], loaded["num_classes"]
        y_true, preds = loaded["y_true"], loaded["predictions"]
        N, pool = len(y_true), list(preds.keys())
        sizes = make_test_sizes(loaded["n_test_min"], N, NUM_SIZES)
        t_ds, n_new = time.time(), 0
        for size in sizes:
            for trial in range(TRIALS_PER_SIZE):
                if (name, size, trial) in done:
                    continue
                seed = int(size * 100 + trial)
                idx = build_subsample_indices(N, size, seed)
                yt = y_true[idx]
                sub = {m: preds[m][idx] for m in pool}
                rec = {"dataset": name, "size": int(size), "trial": trial, "seed": seed,
                       "subsample_hash": hash_indices(idx), "num_classes": C, "num_models": len(pool),
                       "macro_f1": {}, "per_class_f1": {}, "time": {}}
                t0 = time.time()
                y, experts, iters = dbcc_predict(sub, C, em=True, max_iter=MAX_EM_ITER, seed=seed, return_experts=True)
                rec["time"]["dbcc"] = time.time() - t0
                perf = summarize_method_performance(yt, y, C)
                rec["macro_f1"]["dbcc"] = perf["macro_f1"]
                rec["per_class_f1"]["dbcc"] = {int(k): v["f1"] for k, v in perf["per_class"].items()}
                rec["dbcc_experts"] = {int(k): v for k, v in experts.items()}
                rec["dbcc_em_iters"] = iters
                if RECORD_STAGE1_TOO:
                    t0 = time.time()
                    y1 = dbcc_predict(sub, C, em=False, seed=seed)
                    rec["time"]["dbcc_stage1"] = time.time() - t0
                    perf1 = summarize_method_performance(yt, y1, C)
                    rec["macro_f1"]["dbcc_stage1"] = perf1["macro_f1"]
                    rec["per_class_f1"]["dbcc_stage1"] = {int(k): v["f1"] for k, v in perf1["per_class"].items()}
                out.write(json.dumps(rec) + "\n"); out.flush(); n_new += 1
        print(f"  {name}: {n_new} new record(s), {len(sizes)} sizes x {TRIALS_PER_SIZE} trials, {time.time()-t_ds:.0f}s")
print("Done ->", OUTPUT_JSONL)